# PlastiCrete AI — Module 1: Multi-Target Ensemble Training

**Predicts 7 concrete properties from 8 mix-design inputs using XGBoost + Random Forest + DNN ensemble.**

Targets: Compressive Strength · Flexural Strength · Split Tensile · Density · Water Absorption · Thermal Conductivity · Durability Index

---
**Instructions:**
1. Runtime → Change runtime type → **T4 GPU** (free tier)
2. Upload `PlastiCreteAI_M1_Final_2000.csv` when prompted in Section 2
3. Run all cells top-to-bottom (`Runtime → Run all`)
4. Download `plasticrete_m1_models.zip` from the last cell

## 1. Install Dependencies

In [ ]:
%%capture
!pip install xgboost scikit-learn shap pandas numpy matplotlib seaborn joblib torch

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, json, zipfile, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor

import xgboost as xgb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')
print('All imports successful.')

## 2. Load Dataset

In [ ]:
from google.colab import files

print('Upload PlastiCreteAI_M1_Final_2000.csv')
uploaded = files.upload()
CSV_NAME = list(uploaded.keys())[0]
print(f'Uploaded: {CSV_NAME}')

In [ ]:
df_raw = pd.read_csv(CSV_NAME)

# Clean column names — strip whitespace, fix encoding artefacts
df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.replace('Â°', '°', regex=False)
    .str.replace('Â³', '³', regex=False)
    .str.replace('Â·', '·', regex=False)
)

# Canonical column mapping
COL_MAP = {
    'Plastic type':                  'plastic_type',
    'Replacement %':                 'replacement_pct',
    'Particle size (mm)':            'particle_size_mm',
    'w/c ratio':                     'wc_ratio',
    'Additive type':                 'additive_type',
    'Additive %':                    'additive_pct',
    'Curing temperature (°C)':       'curing_temp_c',
    'Curing duration (days)':        'curing_days',
    'Compressive strength (MPa)':    'compressive_strength_mpa',
    'Flexural strength (MPa)':       'flexural_strength_mpa',
    'Split tensile strength (MPa)':  'split_tensile_mpa',
    'Density (kg/m³)':               'density_kgm3',
    'Water absorption (%)':          'water_absorption_pct',
    'Thermal conductivity (W/m·K)':  'thermal_conductivity_wm',
    'Durability index':              'durability_index',
}
df_raw.rename(columns={k: v for k, v in COL_MAP.items() if k in df_raw.columns}, inplace=True)

INPUT_COLS = ['plastic_type','replacement_pct','particle_size_mm','wc_ratio',
              'additive_type','additive_pct','curing_temp_c','curing_days']
TARGET_COLS = ['compressive_strength_mpa','flexural_strength_mpa','split_tensile_mpa',
               'density_kgm3','water_absorption_pct','thermal_conductivity_wm','durability_index']

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(3)

## 3. Exploratory Data Analysis

In [ ]:
print('=== Target Coverage ===')
for t in TARGET_COLS:
    n = df_raw[t].notna().sum()
    pct = n / len(df_raw) * 100
    print(f'  {t:<35} {n:>5} rows ({pct:.1f}%)')

print('\n=== Plastic Types ===')
print(df_raw['plastic_type'].value_counts().to_string())

print('\n=== Additive Types ===')
print(df_raw['additive_type'].value_counts().to_string())

print('\n=== Numeric Summary ===')
df_raw[INPUT_COLS[1:] + TARGET_COLS].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(TARGET_COLS):
    axes[i].hist(df_raw[col].dropna(), bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=10)
    axes[i].set_xlabel('Value')
axes[-1].set_visible(False)
plt.suptitle('Target Variable Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = df_raw.groupby('plastic_type')['compressive_strength_mpa'].median().sort_values(ascending=False).index
df_raw.boxplot(column='compressive_strength_mpa', by='plastic_type',
               ax=axes[0], boxprops=dict(color='steelblue'))
axes[0].set_title('Compressive Strength by Plastic Type')
axes[0].set_xlabel('Plastic Type')
axes[0].set_ylabel('CS (MPa)')

scatter = axes[1].scatter(df_raw['replacement_pct'], df_raw['compressive_strength_mpa'],
                          c=df_raw['wc_ratio'], cmap='RdYlGn_r', alpha=0.4, s=12)
plt.colorbar(scatter, ax=axes[1], label='w/c ratio')
axes[1].set_xlabel('Replacement %')
axes[1].set_ylabel('Compressive Strength (MPa)')
axes[1].set_title('CS vs Replacement % (coloured by w/c ratio)')

plt.suptitle('')
plt.tight_layout()
plt.savefig('eda_cs_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
numeric_cols = ['replacement_pct','particle_size_mm','wc_ratio','additive_pct',
                'curing_temp_c','curing_days'] + TARGET_COLS
corr = df_raw[numeric_cols].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature & Target Correlation Matrix', fontsize=12)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing & Feature Engineering

In [ ]:
df = df_raw.copy()

# ── Normalise additive_type strings ──────────────────────────────────────────
ADDITIVE_MAP = {
    'none': 'none', 'silica fume': 'silica_fume', 'silica_fume': 'silica_fume',
    'fly ash': 'fly_ash', 'fly_ash': 'fly_ash',
    'ggbs': 'ggbs', 'GGBS': 'ggbs',
    'fibres': 'fibres', 'fiber': 'fibres',
}
df['additive_type'] = df['additive_type'].str.strip().map(
    lambda v: ADDITIVE_MAP.get(str(v).strip(), 'none')
)

# ── Feature engineering ───────────────────────────────────────────────────────
df['log_particle_size']     = np.log1p(df['particle_size_mm'])
df['log_curing_days']       = np.log1p(df['curing_days'])
df['replacement_x_wc']      = df['replacement_pct'] * df['wc_ratio']
df['replacement_x_additive']= df['replacement_pct'] * df['additive_pct']
df['is_plain']               = (df['replacement_pct'] == 0).astype(float)

PLASTIC_CATS  = ['PET','HDPE','LDPE','PVC','PP','Mixed','None']
ADDITIVE_CATS = ['fly_ash','silica_fume','ggbs','fibres','none']

# Ordinal encode categoricals
plastic_enc  = OrdinalEncoder(categories=[PLASTIC_CATS],
                               handle_unknown='use_encoded_value', unknown_value=-1)
additive_enc = OrdinalEncoder(categories=[ADDITIVE_CATS],
                               handle_unknown='use_encoded_value', unknown_value=-1)

df['plastic_type_enc']  = plastic_enc.fit_transform(df[['plastic_type']])
df['additive_type_enc'] = additive_enc.fit_transform(df[['additive_type']])

FEATURE_COLS = [
    'plastic_type_enc','replacement_pct','particle_size_mm','wc_ratio',
    'additive_type_enc','additive_pct','curing_temp_c','curing_days',
    'log_particle_size','log_curing_days','replacement_x_wc',
    'replacement_x_additive','is_plain',
]

X = df[FEATURE_COLS].values.astype(np.float32)
Y = df[TARGET_COLS].values.astype(np.float32)

print(f'X shape: {X.shape}')
print(f'Y shape: {Y.shape}')
print(f'Features: {FEATURE_COLS}')

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.15, random_state=SEED
)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.12, random_state=SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

## 5. Train XGBoost (per-target)

In [ ]:
XGB_PARAMS = dict(
    n_estimators=800, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0, random_state=SEED,
    tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
    early_stopping_rounds=50, eval_metric='rmse',
)

xgb_models = {}
xgb_preds_test = np.zeros_like(Y_test)

print('Training XGBoost (one model per target)...')
for i, target in enumerate(TARGET_COLS):
    y_tr = Y_train[:, i]
    y_val_t = Y_val[:, i]
    mask_tr  = ~np.isnan(y_tr)
    mask_val = ~np.isnan(y_val_t)

    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(
        X_train[mask_tr], y_tr[mask_tr],
        eval_set=[(X_val[mask_val], y_val_t[mask_val])],
        verbose=False,
    )
    xgb_models[target] = model

    y_te = Y_test[:, i]
    mask_te = ~np.isnan(y_te)
    pred = model.predict(X_test[mask_te])
    xgb_preds_test[mask_te, i] = pred
    r2 = r2_score(y_te[mask_te], pred)
    mae = mean_absolute_error(y_te[mask_te], pred)
    print(f'  {target:<35} R²={r2:.4f}  MAE={mae:.4f}')

print('XGBoost training complete.')

## 6. Train Random Forest

In [ ]:
rf_models = {}
rf_preds_test = np.zeros_like(Y_test)

print('Training Random Forest (one model per target)...')
for i, target in enumerate(TARGET_COLS):
    y_tr = Y_train[:, i]
    mask_tr = ~np.isnan(y_tr)
    y_te    = Y_test[:, i]
    mask_te = ~np.isnan(y_te)

    model = RandomForestRegressor(
        n_estimators=500, max_depth=None, min_samples_leaf=2,
        max_features=0.6, n_jobs=-1, random_state=SEED
    )
    model.fit(X_train[mask_tr], y_tr[mask_tr])
    rf_models[target] = model

    pred = model.predict(X_test[mask_te])
    rf_preds_test[mask_te, i] = pred
    r2  = r2_score(y_te[mask_te], pred)
    mae = mean_absolute_error(y_te[mask_te], pred)
    print(f'  {target:<35} R²={r2:.4f}  MAE={mae:.4f}')

print('Random Forest training complete.')

## 7. Train DNN (with MC Dropout)

In [ ]:
class PlastiCreteDNN(nn.Module):
    def __init__(self, n_features: int, n_targets: int, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),  nn.SiLU(),
            nn.Linear(64, n_targets),
        )

    def forward(self, x):
        return self.net(x)


def masked_mse(pred, target):
    """MSE only on non-NaN positions — handles partial target coverage."""
    mask = ~torch.isnan(target)
    if mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True)
    return ((pred[mask] - target[mask]) ** 2).mean()


n_feat    = X_train_s.shape[1]
n_targets = len(TARGET_COLS)

dnn = PlastiCreteDNN(n_feat, n_targets, dropout=0.2).to(DEVICE)
print(dnn)
print(f'Parameters: {sum(p.numel() for p in dnn.parameters()):,}')

In [ ]:
# ── Scale targets for DNN ────────────────────────────────────────────────────
target_scaler = StandardScaler()
Y_train_s = target_scaler.fit_transform(
    np.where(np.isnan(Y_train), 0, Y_train)
).astype(np.float32)
Y_train_s = np.where(np.isnan(Y_train), np.nan, Y_train_s)

Y_val_s = target_scaler.transform(
    np.where(np.isnan(Y_val), 0, Y_val)
).astype(np.float32)
Y_val_s = np.where(np.isnan(Y_val), np.nan, Y_val_s)

# Build DataLoaders
train_ds = TensorDataset(
    torch.tensor(X_train_s), torch.tensor(Y_train_s)
)
val_ds = TensorDataset(
    torch.tensor(X_val_s), torch.tensor(Y_val_s)
)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256)

optimizer = optim.AdamW(dnn.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150, eta_min=1e-6)

EPOCHS     = 150
best_val   = float('inf')
best_state = None
train_losses, val_losses = [], []

print(f'Training DNN for {EPOCHS} epochs on {DEVICE}...')
for epoch in range(1, EPOCHS + 1):
    dnn.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = masked_mse(dnn(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(dnn.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    dnn.eval()
    with torch.no_grad():
        vl = sum(
            masked_mse(dnn(xb.to(DEVICE)), yb.to(DEVICE)).item()
            for xb, yb in val_loader
        )
    train_losses.append(epoch_loss / len(train_loader))
    val_losses.append(vl / len(val_loader))

    if vl < best_val:
        best_val   = vl
        best_state = {k: v.cpu().clone() for k, v in dnn.state_dict().items()}

    if epoch % 25 == 0:
        print(f'  Epoch {epoch:3d}/{EPOCHS}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}')

dnn.load_state_dict(best_state)
print(f'DNN training complete. Best val loss: {best_val:.4f}')

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Masked MSE')
plt.title('DNN Training Curves')
plt.legend()
plt.tight_layout()
plt.savefig('dnn_loss_curve.png', dpi=150)
plt.show()

In [ ]:
dnn.eval()
with torch.no_grad():
    dnn_preds_scaled = dnn(torch.tensor(X_test_s).to(DEVICE)).cpu().numpy()

# Inverse-transform to original scale
dnn_preds_test = target_scaler.inverse_transform(dnn_preds_scaled)

print('DNN test predictions (per-target R²):')
for i, target in enumerate(TARGET_COLS):
    y_te    = Y_test[:, i]
    mask_te = ~np.isnan(y_te)
    r2  = r2_score(y_te[mask_te], dnn_preds_test[mask_te, i])
    mae = mean_absolute_error(y_te[mask_te], dnn_preds_test[mask_te, i])
    print(f'  {target:<35} R²={r2:.4f}  MAE={mae:.4f}')

## 8. Ensemble (Weighted Average)

In [ ]:
# Compute per-target R² on validation set to derive weights
def get_val_r2(models_dict, X_v, Y_v):
    r2s = {}
    for i, t in enumerate(TARGET_COLS):
        y_v = Y_v[:, i]
        mask = ~np.isnan(y_v)
        if mask.sum() > 5:
            pred = models_dict[t].predict(X_v[mask])
            r2s[t] = max(r2_score(y_v[mask], pred), 0)
        else:
            r2s[t] = 0.0
    return r2s

xgb_r2 = get_val_r2(xgb_models, X_val, Y_val)
rf_r2  = get_val_r2(rf_models,  X_val, Y_val)

dnn.eval()
with torch.no_grad():
    dnn_val_scaled = dnn(torch.tensor(X_val_s).to(DEVICE)).cpu().numpy()
dnn_val = target_scaler.inverse_transform(dnn_val_scaled)
dnn_r2  = {}
for i, t in enumerate(TARGET_COLS):
    y_v  = Y_val[:, i]
    mask = ~np.isnan(y_v)
    dnn_r2[t] = max(r2_score(y_v[mask], dnn_val[mask, i]), 0) if mask.sum() > 5 else 0.0

# Weighted softmax ensemble
ENS_WEIGHTS = {}
ens_preds_test = np.zeros_like(Y_test)

for i, t in enumerate(TARGET_COLS):
    w_xgb = xgb_r2[t] ** 2
    w_rf  = rf_r2[t]  ** 2
    w_dnn = dnn_r2[t] ** 2
    total = w_xgb + w_rf + w_dnn + 1e-9
    w_xgb /= total; w_rf /= total; w_dnn /= total
    ENS_WEIGHTS[t] = {'xgb': w_xgb, 'rf': w_rf, 'dnn': w_dnn}

    y_te  = Y_test[:, i]
    mask  = ~np.isnan(y_te)
    ens   = (w_xgb * xgb_preds_test[mask, i] +
             w_rf  * rf_preds_test[mask, i]  +
             w_dnn * dnn_preds_test[mask, i])
    ens_preds_test[mask, i] = ens

print('=== FINAL ENSEMBLE TEST RESULTS ===')
results = {}
for i, t in enumerate(TARGET_COLS):
    y_te  = Y_test[:, i]
    mask  = ~np.isnan(y_te)
    r2    = r2_score(y_te[mask], ens_preds_test[mask, i])
    mae   = mean_absolute_error(y_te[mask], ens_preds_test[mask, i])
    rmse  = mean_squared_error(y_te[mask], ens_preds_test[mask, i]) ** 0.5
    results[t] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    w = ENS_WEIGHTS[t]
    print(f'  {t:<35} R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  '
          f'[XGB={w["xgb"]:.2f} RF={w["rf"]:.2f} DNN={w["dnn"]:.2f}]')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
for i, t in enumerate(TARGET_COLS):
    y_te  = Y_test[:, i]
    mask  = ~np.isnan(y_te)
    true  = y_te[mask]
    pred  = ens_preds_test[mask, i]
    r2    = results[t]['R2']
    axes[i].scatter(true, pred, alpha=0.35, s=10, color='steelblue')
    lo, hi = min(true.min(), pred.min()), max(true.max(), pred.max())
    axes[i].plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect')
    axes[i].set_title(f'{t.replace("_", " ").title()}\nR²={r2:.3f}', fontsize=9)
    axes[i].set_xlabel('Actual')
    axes[i].set_ylabel('Predicted')
axes[-1].set_visible(False)
plt.suptitle('Ensemble: Predicted vs Actual (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig('ensemble_pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
r2_vals = [results[t]['R2'] for t in TARGET_COLS]
labels  = [t.replace('_', '\n') for t in TARGET_COLS]

colors = ['#2ecc71' if r >= 0.8 else '#f39c12' if r >= 0.6 else '#e74c3c' for r in r2_vals]

plt.figure(figsize=(12, 5))
bars = plt.bar(labels, r2_vals, color=colors, edgecolor='white', linewidth=0.8)
plt.axhline(0.80, color='green',  linestyle='--', alpha=0.6, label='R²=0.80 threshold')
plt.axhline(0.60, color='orange', linestyle='--', alpha=0.6, label='R²=0.60 threshold')
for bar, val in zip(bars, r2_vals):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontsize=9)
plt.ylim(-0.1, 1.05)
plt.ylabel('R² Score')
plt.title('Ensemble R² per Target (Test Set)')
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig('r2_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. SHAP Feature Importance (Compressive Strength)

In [ ]:
TARGET_FOR_SHAP = 'compressive_strength_mpa'
xgb_cs = xgb_models[TARGET_FOR_SHAP]

explainer = shap.TreeExplainer(xgb_cs)
sample_idx = np.random.choice(len(X_test), min(300, len(X_test)), replace=False)
shap_values = explainer.shap_values(X_test[sample_idx])

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test[sample_idx],
    feature_names=FEATURE_COLS,
    show=False, plot_type='bar'
)
plt.title(f'SHAP Feature Importance — {TARGET_FOR_SHAP}')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test[sample_idx],
    feature_names=FEATURE_COLS,
    show=False
)
plt.title(f'SHAP Beeswarm — {TARGET_FOR_SHAP}')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Models & Download

In [ ]:
os.makedirs('plasticrete_m1_models', exist_ok=True)

# Save XGBoost models
os.makedirs('plasticrete_m1_models/xgboost', exist_ok=True)
for t, m in xgb_models.items():
    m.save_model(f'plasticrete_m1_models/xgboost/{t}.json')

# Save Random Forest models
os.makedirs('plasticrete_m1_models/random_forest', exist_ok=True)
for t, m in rf_models.items():
    joblib.dump(m, f'plasticrete_m1_models/random_forest/{t}.joblib')

# Save DNN
os.makedirs('plasticrete_m1_models/dnn', exist_ok=True)
torch.save(dnn.state_dict(), 'plasticrete_m1_models/dnn/dnn_weights.pt')

# Save preprocessors
joblib.dump(scaler,         'plasticrete_m1_models/feature_scaler.joblib')
joblib.dump(target_scaler,  'plasticrete_m1_models/target_scaler.joblib')
joblib.dump(plastic_enc,    'plasticrete_m1_models/plastic_encoder.joblib')
joblib.dump(additive_enc,   'plasticrete_m1_models/additive_encoder.joblib')

# Save metadata
metadata = {
    'feature_cols':  FEATURE_COLS,
    'target_cols':   TARGET_COLS,
    'ensemble_weights': ENS_WEIGHTS,
    'test_metrics':  results,
    'plastic_cats':  PLASTIC_CATS,
    'additive_cats': ADDITIVE_CATS,
    'dnn_architecture': {'n_features': n_feat, 'n_targets': n_targets, 'dropout': 0.2},
}
with open('plasticrete_m1_models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Zip everything
with zipfile.ZipFile('plasticrete_m1_models.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk('plasticrete_m1_models'):
        for file in files:
            filepath = os.path.join(root, file)
            zf.write(filepath)
    for png in ['target_distributions.png','eda_cs_analysis.png','correlation_matrix.png',
                'dnn_loss_curve.png','ensemble_pred_vs_actual.png','r2_bar_chart.png',
                'shap_importance.png','shap_beeswarm.png']:
        if os.path.exists(png):
            zf.write(png)

print('All models and plots saved to plasticrete_m1_models.zip')
print()
print('=== FINAL SUMMARY ===')
for t, m in results.items():
    print(f'  {t:<35} R²={m["R2"]:.4f}  MAE={m["MAE"]:.4f}  RMSE={m["RMSE"]:.4f}')

In [ ]:
files.download('plasticrete_m1_models.zip')
print('Download started. Check your browser downloads.')

## 11. Quick Inference Demo

Run a single prediction using the trained ensemble — verifies the models work end-to-end.

In [ ]:
def predict_single(
    plastic_type: str,
    replacement_pct: float,
    particle_size_mm: float,
    wc_ratio: float,
    additive_type: str,
    additive_pct: float,
    curing_temp_c: float,
    curing_days: float,
) -> pd.DataFrame:
    """Predict all 7 targets for a single mix design."""
    row = pd.DataFrame([{
        'plastic_type': plastic_type, 'replacement_pct': replacement_pct,
        'particle_size_mm': particle_size_mm, 'wc_ratio': wc_ratio,
        'additive_type': additive_type, 'additive_pct': additive_pct,
        'curing_temp_c': curing_temp_c, 'curing_days': curing_days,
    }])
    row['additive_type'] = row['additive_type'].map(lambda v: ADDITIVE_MAP.get(str(v).strip(), 'none'))
    row['plastic_type_enc']  = plastic_enc.transform(row[['plastic_type']])
    row['additive_type_enc'] = additive_enc.transform(row[['additive_type']])
    row['log_particle_size']      = np.log1p(row['particle_size_mm'])
    row['log_curing_days']        = np.log1p(row['curing_days'])
    row['replacement_x_wc']       = row['replacement_pct'] * row['wc_ratio']
    row['replacement_x_additive'] = row['replacement_pct'] * row['additive_pct']
    row['is_plain']               = (row['replacement_pct'] == 0).astype(float)

    X_in  = row[FEATURE_COLS].values.astype(np.float32)
    X_ins = scaler.transform(X_in)

    preds = {}
    for i, t in enumerate(TARGET_COLS):
        w = ENS_WEIGHTS[t]
        xgb_p = xgb_models[t].predict(X_in)[0]
        rf_p  = rf_models[t].predict(X_in)[0]
        dnn.eval()
        with torch.no_grad():
            dnn_scaled = dnn(torch.tensor(X_ins).to(DEVICE)).cpu().numpy()
        dnn_p = target_scaler.inverse_transform(dnn_scaled)[0, i]
        preds[t] = w['xgb'] * xgb_p + w['rf'] * rf_p + w['dnn'] * dnn_p

    return pd.DataFrame([preds]).T.rename(columns={0: 'Predicted'})


# Example: PET, 2% replacement, 5mm particle, w/c=0.4, silica fume 10%, 20°C, 28 days
result = predict_single(
    plastic_type='PET', replacement_pct=2.0, particle_size_mm=5.0,
    wc_ratio=0.40, additive_type='silica fume', additive_pct=10.0,
    curing_temp_c=20.0, curing_days=28.0
)
print('=== Single Mix Prediction ===')
print(result.round(3).to_string())